Import pandas and load movies file

In [ ]:
import pandas as pd

movies = pd.read_csv("./data/movies.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


get list of unique genres

In [7]:
unique_genres = set()

for genre_list in movies["genres"].str.split("|"):
    unique_genres.update(genre_list)

print(len(unique_genres))
print(sorted(unique_genres))

20
['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


So the feature space will be ~20 columns. Each movie will be a 20 dimensional vector

Then we one-hot encode the genres

In [8]:
genre_matrix = movies["genres"].str.get_dummies(sep="|")

genre_matrix.head()
genre_matrix.shape

(86537, 20)

cosine similarity

In [9]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(genre_matrix)

def recommend(movie_title, movies, genre_matrix, model, n=5):
    idx = movies[movies["title"] == movie_title].index[0]
    
    distances, indices = model.kneighbors(
        genre_matrix.iloc[idx].values.reshape(1, -1),
        n_neighbors=n + 1
    )
    
    neighbor_indices = indices.flatten()[1:]  # skip itself
    return movies.iloc[neighbor_indices][["title", "genres"]]

recommend("Toy Story (1995)", movies, genre_matrix, knn)


/Users/annmathenge/Projects/machine_learning_lab/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


,title,genres
61975,UglyDolls (2019),Adventure|Animation|Children|Comedy|Fantasy
79965,Christmas in Tattertown (1988),Adventure|Animation|Children|Comedy|Fantasy
70762,The SpongeBob Movie: Sponge on the Run (2020),Adventure|Animation|Children|Comedy|Fantasy
23239,The Magic Crystal (2011),Adventure|Animation|Children|Comedy|Fantasy
80649,Riverdance: The Animated Adventure (2021),Adventure|Animation|Children|Comedy|Fantasy


Inspect tags

In [ ]:
tags = pd.read_csv("./data/tags.csv")

tags.shape
tags["tag"].isna().sum()
tags["movieId"].nunique()
tags["tag"].value_counts().head(20) 

tag
sci-fi                14319
atmospheric           12172
action                10683
comedy                10161
surreal                9142
funny                  9094
visually appealing     8890
twist ending           8325
thought-provoking      7727
dark comedy            7659
based on a book        7584
dystopia               6975
cinematography         6473
social commentary      6369
romance                6338
violence               6300
stylized               6163
psychology             6111
fantasy                6082
murder                 5955
Name: count, dtype: int64